# Crop Classification (Sen4AgriNet, Catalonia)

This notebook keeps a simple Random Forest baseline and adds an interpretability layer that maps Sen4AgriNet numeric class ids to crop names using the official Catalonia taxonomy logic from `mappings_cat.py` + `encodings_en.py` in `S4A-Models`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix


## Label Mapping and Crop Names

The dictionary below is adapted from:
- `utils/settings/mappings/mappings_cat.py` (Catalonia taxonomy logic)
- `utils/settings/mappings/encodings_en.py` (English class name to numeric code)
- `utils/settings/config.py` (11 selected experimental classes)

For interpretability in this notebook, we map numeric class ids directly to readable crop names.


In [ ]:
# 11 most frequent selected classes used in the Sen4AgriNet-Models experiments
# (from utils/settings/config.py)
SELECTED_CLASS_IDS = [110, 120, 140, 150, 160, 170, 330, 435, 438, 510, 770]

# Numeric class id -> crop name
LABEL_TO_CROP_NAME = {
    110: "wheat",
    120: "maize",
    140: "sorghum",
    150: "barley",
    160: "rye",
    170: "oats",
    330: "grapes",
    435: "rapeseed",
    438: "sunflower",
    510: "potatoes",
    770: "peas",
}


def label_display(label):
    """Return explicit display label such as '330 | grapes' when known."""
    label = int(label)
    return f"{label} | {LABEL_TO_CROP_NAME.get(label, 'UNMAPPED')}"


print("Catalonia mapping dictionary (selected classes):")
for k in sorted(LABEL_TO_CROP_NAME):
    print(f"  {k}: {LABEL_TO_CROP_NAME[k]}")


In [ ]:
def _collect_present_labels_from_globals(g):
    """Best-effort collection of labels present in the current notebook state."""
    candidates = []

    # Common variable names used in crop-classification notebooks
    for key in [
        'y', 'y_all', 'labels', 'y_filtered', 'labels_filtered',
        'y_train', 'y_test', 'y_val', 'y_pred', 'y_pred_test'
    ]:
        if key in g and g[key] is not None:
            arr = np.asarray(g[key]).ravel()
            if arr.size:
                candidates.append(arr)

    if not candidates:
        return np.array([], dtype=int)

    merged = np.concatenate(candidates)
    merged = merged[pd.notna(merged)]

    try:
        return np.unique(merged.astype(int))
    except Exception:
        # If conversion fails, return empty set to avoid breaking baseline
        return np.array([], dtype=int)


present_labels = _collect_present_labels_from_globals(globals())

if present_labels.size == 0:
    print(
        "No label arrays detected yet. "
        "Run your baseline data-prep/training cells first, then re-run this section."
    )
else:
    mapping_rows = []
    for code in sorted(present_labels.tolist()):
        name = LABEL_TO_CROP_NAME.get(code)
        mapping_rows.append({
            "label_code": code,
            "crop_name": name if name is not None else "UNMAPPED",
            "mapped": name is not None,
        })

    mapping_df = pd.DataFrame(mapping_rows)
    display(mapping_df)

    mapped_codes = mapping_df.loc[mapping_df["mapped"], "label_code"].tolist()
    unmapped_codes = mapping_df.loc[~mapping_df["mapped"], "label_code"].tolist()

    print(f"Mapped labels in current dataset: {mapped_codes}")
    if unmapped_codes:
        print(f"Unmapped labels in current dataset: {unmapped_codes}")
    else:
        print("All labels in the current dataset were mapped.")


In [ ]:
def label_count_table_with_names(y_values):
    """Return count table with both numeric codes and crop names."""
    s = pd.Series(np.asarray(y_values).ravel().astype(int), name="label_code")
    out = (
        s.value_counts(dropna=False)
        .rename_axis("label_code")
        .reset_index(name="count")
        .sort_values("label_code")
        .reset_index(drop=True)
    )
    out["crop_name"] = out["label_code"].map(LABEL_TO_CROP_NAME).fillna("UNMAPPED")
    out["label_display"] = out["label_code"].apply(label_display)
    return out[["label_code", "crop_name", "label_display", "count"]]


def classification_report_with_names(y_true, y_pred):
    """Classification report with explicit 'code | crop' class names when possible."""
    y_true = np.asarray(y_true).ravel().astype(int)
    y_pred = np.asarray(y_pred).ravel().astype(int)

    labels = sorted(np.unique(np.concatenate([y_true, y_pred])).tolist())
    target_names = [label_display(x) for x in labels]

    print(classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0,
    ))


def plot_confusion_matrix_with_names(y_true, y_pred, normalize=None, cmap="Blues"):
    """Confusion matrix with explicit code/name axis labels."""
    y_true = np.asarray(y_true).ravel().astype(int)
    y_pred = np.asarray(y_pred).ravel().astype(int)

    labels = sorted(np.unique(np.concatenate([y_true, y_pred])).tolist())
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize=normalize)
    disp_labels = [label_display(x) for x in labels]

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(len(disp_labels)),
        yticks=np.arange(len(disp_labels)),
        xticklabels=disp_labels,
        yticklabels=disp_labels,
        xlabel="Predicted label",
        ylabel="True label",
        title="Confusion matrix" + (" (normalized)" if normalize else ""),
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    plt.tight_layout()
    plt.show()


### Usage in your existing baseline cells

- Replace old label-count prints with: `label_count_table_with_names(y_train)` (and/or `y_test`, `y_filtered`).
- Replace plain classification report with: `classification_report_with_names(y_test, y_pred)`.
- Replace confusion matrix plotting with: `plot_confusion_matrix_with_names(y_test, y_pred, normalize=None)`.

This keeps the baseline model unchanged (Random Forest) and only improves interpretability.


---

**Note:** The baseline model was first trained on numeric class ids. This mapping step improves interpretability by linking class ids to crop names where the official Catalonia taxonomy mapping is available.
